In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lifelines import KaplanMeierFitter, CoxPHFitter, ExponentialFitter, WeibullFitter
from lifelines.plotting import plot_lifetimes
from ISLP import load_data

## Simulated Data

In [ ]:
CURRENT_TIME = 15

n_samples = 25
actual_lifetimes = np.array([
    np.random.exponential(12)
    if (np.random.uniform() < 0.5) else np.random.exponential(2)
    for _ in range(n_samples)
])
observed_lifetimes = np.minimum(actual_lifetimes, CURRENT_TIME)
death_observed = actual_lifetimes < CURRENT_TIME

ax = plot_lifetimes(
    observed_lifetimes,
    event_observed=death_observed,
    sort_by_duration=False,
)

ax.set_xlim(0, 25)
ax.vlines(CURRENT_TIME, 0, 30, lw=2, linestyles='--')
ax.set_xlabel("time")
plt.tight_layout();

## Brain Cancer Survival

In [ ]:
cancer = load_data("BrainCancer")
cancer.head()

In [ ]:
# Fit Kaplan-Meier Survival curve

male = cancer["sex"] == "Male"

km_male = KaplanMeierFitter()
km_male = km_male.fit(cancer[male]["time"], event_observed=cancer[male]["status"])

km_female = KaplanMeierFitter()
km_female = km_female.fit(cancer[~male]["time"], event_observed=cancer[~male]["status"])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
km_male.plot_survival_function(label="male", ax=ax[0])
km_female.plot_survival_function(label="female", ax=ax[1])
fig.tight_layout();

In [ ]:
# Parametric models

male = cancer["sex"] == "Male"

durations = cancer[male]["time"]
event_observed = cancer[male]["status"]

kmf = KaplanMeierFitter()
kmf = kmf.fit(cancer[male]["time"], event_observed=event_observed)

wdf = WeibullFitter()
wdf = wdf.fit(cancer[male]["time"], event_observed=event_observed)

expf = ExponentialFitter()
expf = expf.fit(cancer[male]["time"], event_observed=event_observed)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
kmf.plot_survival_function(label="KaplanMeier", ax=ax[0])
wdf.plot_survival_function(label="Weibull", ax=ax[1], xlabel="timeline")
expf.plot_survival_function(label="Exponential", ax=ax[2], xlabel="timeline")
fig.tight_layout();

## Regression Models

In [ ]:
churn = pd.read_csv('churn.csv')

churn.head()

In [ ]:
churn_df = churn[["minutes_customer_support", "product_payroll", "months_active", "churned"]]
payroll_dummies = pd.get_dummies(churn["product_payroll"], drop_first=True).astype(float)
churn_df = pd.concat([churn_df, payroll_dummies], axis=1).drop(columns=["product_payroll"])

cph = CoxPHFitter()
cph = cph.fit(churn_df, duration_col="months_active", event_col="churned")

In [ ]:
cph.summary

In [ ]:
cph.plot_partial_effects_on_outcome(
    covariates="minutes_customer_support",
    values=[0, 10, 20, 30],
)

In [ ]:
cph.plot_partial_effects_on_outcome(
    covariates=["Free-Trial", "No"],
    values=[[0, 0], [1, 0], [0, 1],],
)

In [ ]:
# Prediction

X_test = pd.DataFrame([[20, 1, 0]], columns=["minutes_customer_support", "Free-Trial", "No"])
ax = cph.plot_partial_effects_on_outcome(
    covariates=["minutes_customer_support", "Free-Trial", "No"],
    values=X_test,
)
ax.legend(fontsize="small")

In [ ]:
cph.predict_median(X=X_test)

In [ ]:
times_to_churn = [cph.predict_percentile(X=X_test, p=p) for p in np.random.uniform(size=1000)]
times_to_churn = [t for t in times_to_churn if not np.isinf(t)]

np.median(times_to_churn)

In [ ]:
plt.hist(times_to_churn, density=True);